# V7_0_N01 — Start With the Decision

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft v0.1. This notebook supports learning and review; it does not authorize an operational decision.

## Learning outcomes
By the end, you can: (1) define a public decision before choosing a model; (2) name the accountable owner and lawful intervention; (3) state the decision horizon and error trade-offs; (4) define abstention and escalation; and (5) produce a machine-readable decision contract.

## 1. Why the decision comes first
A forecast is useful only when a named institution can act within the forecast horizon. We therefore separate: **signal**, **analytical interpretation**, **recommendation**, **authorization**, and **action**. The model owns none of the last three.

In [1]:
from dataclasses import dataclass, asdict
from typing import Optional
import json

@dataclass
class DecisionContract:
    decision_id: str
    public_problem: str
    accountable_owner: str
    decision: str
    horizon_days: int
    intervention: str
    false_positive_cost: str
    false_negative_cost: str
    abstain_when: str
    escalation_route: str
    review_cycle_days: int

print('DecisionContract ready')

DecisionContract ready


## 2. A worked agricultural contract
The analytical system may identify areas requiring field verification. It does not declare a food-security emergency or allocate resources automatically.

In [2]:
agri = DecisionContract(
 'AG-A05','An agricultural frame may omit or misclassify active holdings',
 'National Statistical Office / Ministry of Agriculture',
 'Which areas should receive frame verification before sample selection?',30,
 'Targeted listing and frame reconciliation',
 'Unnecessary field visit and cost','Biased frame and undercoverage',
 'coverage diagnostics are stale or geographic identifiers fail validation',
 'Survey director → methodology committee',14)
print(json.dumps(asdict(agri),indent=2))

{
  "decision_id": "AG-A05",
  "public_problem": "An agricultural frame may omit or misclassify active holdings",
  "accountable_owner": "National Statistical Office / Ministry of Agriculture",
  "decision": "Which areas should receive frame verification before sample selection?",
  "horizon_days": 30,
  "intervention": "Targeted listing and frame reconciliation",
  "false_positive_cost": "Unnecessary field visit and cost",
  "false_negative_cost": "Biased frame and undercoverage",
  "abstain_when": "coverage diagnostics are stale or geographic identifiers fail validation",
  "escalation_route": "Survey director \u2192 methodology committee",
  "review_cycle_days": 14
}


## 3. Loss is not the same as model error
A false negative can be more consequential than a false positive, or vice versa. The relative cost is a policy judgement to document—not a number for the model to invent.

In [3]:
def expected_policy_loss(fp, fn, cost_fp=1, cost_fn=5):
    return fp*cost_fp + fn*cost_fn
scenarios=[('conservative',18,3),('balanced',10,7),('restrictive',4,14)]
for name,fp,fn in scenarios:
    print(name, expected_policy_loss(fp,fn))

conservative 33
balanced 45
restrictive 74


## 4. Actionability gate
A high accuracy score is insufficient. A candidate passes only if the owner, intervention, horizon, authority, and operational capacity are all present.

In [4]:
def actionability_gate(contract, lawful=True, capacity=True, data_current=True):
    checks={
      'named_owner': bool(contract.accountable_owner.strip()),
      'defined_intervention': bool(contract.intervention.strip()),
      'positive_horizon': contract.horizon_days>0,
      'lawful_authority': lawful, 'operational_capacity':capacity,
      'current_data':data_current}
    return checks, all(checks.values())
checks,passed=actionability_gate(agri)
print(checks,'PASS=',passed)

{'named_owner': True, 'defined_intervention': True, 'positive_horizon': True, 'lawful_authority': True, 'operational_capacity': True, 'current_data': True} PASS= True


## 5. Abstention is a valid output
When required evidence is missing, the safest analytical output may be: *insufficient evidence—refer for review*. This protects decision-makers from false precision.

In [5]:
def disposition(probability, data_quality, low=.35, high=.70):
    if data_quality < .80: return 'ABSTAIN — DATA QUALITY REVIEW'
    if probability >= high: return 'ESCALATE FOR HUMAN REVIEW'
    if probability <= low: return 'ROUTINE MONITORING'
    return 'ABSTAIN — UNCERTAIN BAND'
for p,q in [(.82,.95),(.52,.96),(.20,.93),(.90,.62)]: print(p,q,disposition(p,q))

0.82 0.95 ESCALATE FOR HUMAN REVIEW
0.52 0.96 ABSTAIN — UNCERTAIN BAND
0.2 0.93 ROUTINE MONITORING
0.9 0.62 ABSTAIN — DATA QUALITY REVIEW


## 6. Cross-sector exercise
Complete the contract below for an attendance/dropout early-warning service. Do not write 'the AI system' as the accountable owner.

In [6]:
education = DecisionContract(
 'ED-E01','Persistent absence may precede dropout',
 'District education authority and school safeguarding lead',
 'Which schools or learners require proportionate review and support?',14,
 'Human review followed by appropriate learner support',
 'Unnecessary review or stigmatization','Missed opportunity to prevent disengagement',
 'attendance records are incomplete, delayed, or identity linkage is uncertain',
 'School lead → district safeguarding/education authority',7)
checks,passed=actionability_gate(education)
assert passed
print(json.dumps(asdict(education),indent=2))

{
  "decision_id": "ED-E01",
  "public_problem": "Persistent absence may precede dropout",
  "accountable_owner": "District education authority and school safeguarding lead",
  "decision": "Which schools or learners require proportionate review and support?",
  "horizon_days": 14,
  "intervention": "Human review followed by appropriate learner support",
  "false_positive_cost": "Unnecessary review or stigmatization",
  "false_negative_cost": "Missed opportunity to prevent disengagement",
  "abstain_when": "attendance records are incomplete, delayed, or identity linkage is uncertain",
  "escalation_route": "School lead \u2192 district safeguarding/education authority",
  "review_cycle_days": 7
}


## Knowledge check
1. Why must the decision horizon be defined before model selection?  
2. Who owns the final decision?  
3. Give two reasons to abstain.  
4. Why is accuracy alone insufficient?

## Exact solutions
1. It determines whether data latency and forecast lead time permit intervention.  
2. The named lawful institutional authority, never the model.  
3. Examples: inadequate data quality; uncertain probability band; failed linkage; distribution shift.  
4. Accuracy does not encode consequences, equity, calibration, actionability, authority, or operational capacity.

In [7]:
payload={'schema_version':'1.0','contracts':[asdict(agri),asdict(education)]}
text=json.dumps(payload,sort_keys=True)
assert 'AI system' not in agri.accountable_owner
assert all(c['horizon_days']>0 for c in payload['contracts'])
print('V7_0_N01_COMPLETE_EXECUTION_PASS',len(text))

V7_0_N01_COMPLETE_EXECUTION_PASS 1377
